In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt

from quick_pp.database.objects import Project
from quick_pp.database.db_connector import DBConnector

db_conn = DBConnector()

# Load well from saved file
project_name = "NNS_2-5"
well_name = '2-5-7'
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    df = project.get_all_data()
    well_data = project.get_well_data(well_name)

***
# Log Derived Water Saturation

In [ ]:
from ipywidgets import widgets, interact

from quick_pp.saturation import pickett_plot

focused_data = df.copy()

wells = widgets.SelectMultiple(
    options=['All'] + list(focused_data['WELL_NAME'].unique()),
    value=['All'],
    description='Wells:'
)
m = widgets.FloatSlider(
    value=2,
    min=1,
    max=5,
    step=.1,
    readout_format='.1f'
)
min_rw = widgets.FloatSlider(
    value=.01,
    min=.001,
    max=1.0,
    step=.001,
    readout_format='.3f'
)
min_depth = widgets.FloatSlider(
    value=focused_data.DEPTH.min(),
    min=focused_data.DEPTH.min(),
    max=focused_data.DEPTH.max() - 10,
    step=.1,
    readout_format='.1f'
)
max_depth = widgets.FloatSlider(
    value=focused_data.DEPTH.max(),
    min=focused_data.DEPTH.min() + 10,
    max=focused_data.DEPTH.max(),
    step=.1,
    readout_format='.1f'
)

@interact(wells=wells, m=m, min_rw=min_rw, min_depth=min_depth, max_depth=max_depth)
def param(wells, m, min_rw, min_depth, max_depth):
    if 'All' in wells:
        data = focused_data[(focused_data.DEPTH >= min_depth) & (focused_data.DEPTH <= max_depth)]
    else:
        data = focused_data[(focused_data.WELL_NAME.isin(wells)) & (focused_data.DEPTH >= min_depth) & (focused_data.DEPTH <= max_depth)]
    pickett_plot(data['RT'], data['PHIT'], m=m, min_rw=min_rw, title=f'Pickett Plot for {wells[0]}')

In [ ]:
import numpy as np
from matplotlib import ticker as mticker

from quick_pp.saturation import *
from quick_pp.porosity import *

water_salinity = 15e3
m = 2

temp_grad = estimate_temperature_gradient(well_data['TVD'], 'metric')
rw = estimate_rw_temperature_salinity(temp_grad, water_salinity)
b = estimate_b_waxman_smits(temp_grad, rw)
qv = estimate_qv(well_data.VCLAY, well_data.PHIT, cec_clay=.1)
phit_shale = estimate_shale_porosity(well_data.NPHI, well_data.PHIT)
rt_shale = estimate_rt_shale(well_data.RT, well_data.VCLAY)
qvn = estimate_qvn(well_data.VCLAY, well_data.PHIT, phit_shale)

swt_ws = waxman_smits_saturation(well_data['RT'], rw, well_data.PHIE, B=b, Qv=qvn, m=m)
swt_nws = normalized_waxman_smits_saturation(
    well_data.RT, rw, well_data.PHIT, well_data.VCLAY, phit_shale, rt_shale=3, m=m
)
swt_archie = archie_saturation(well_data.RT, rw, well_data.PHIT, m=m)

print(f'\nEstimated formation temperature: {temp_grad.iloc[-1]} degC')
print(f'Estiamte formation water resistivity: {rw.min()} ohmm')
fig, axes = plt.subplots(3, 1, figsize=(15, 5), sharex=True)
axes[0].plot(well_data['DEPTH'], swt_nws, label='SWT Normalized WS')
axes[0].plot(well_data['DEPTH'], swt_ws, label='SWT WS')
axes[0].plot(well_data['DEPTH'], swt_archie, label='SWT Archie')
axes[0].plot(well_data['DEPTH'], np.ones(len(well_data)), color='black', linestyle='--')
axes[0].set_ylim(0, 2)
axes[0].legend()

axes[1].plot(well_data['DEPTH'], rw, label='RW')
axes[1].set_yscale('log')
axes[1].yaxis.set_minor_formatter(mticker.ScalarFormatter())
axes[1].legend()

axes[2].plot(well_data['DEPTH'], temp_grad, label='Temperature')
axes[2].legend()

fig.tight_layout()

***
# Plot the results

In [ ]:
from quick_pp.plotter.plotter import plotly_log
from quick_pp.core_calibration import *

# Plot individual results
well_data['SWT'] = swt_ws
fig = plotly_log(well_data, well_name=well_name, depth_uom='m')
fig.show(config=dict(scrollZoom=True))

# Apply to all

In [ ]:
from tqdm import tqdm

water_salinity = 15e3
m = 2

for well_name, final_data in tqdm(df.groupby('WELL_NAME')):
    tqdm.write(f'Processing {well_name}: {len(final_data)} rows')
    # NOTE: In certain wells, the resistivity logs seemed to have a factor of 10 discrepancy indicating fresh water
    # environment, which is not the case.
    final_data['RT_EDIT'] = final_data.RT
    # if well_name in ['2-5-3', '2-5-6', '2-5-7']:
    #     final_data['RT_EDIT'] = final_data.RT / 10

    temp_grad = estimate_temperature_gradient(final_data['TVD'], 'metric')
    rw = estimate_rw_temperature_salinity(temp_grad, water_salinity)
    b = estimate_b_waxman_smits(temp_grad, rw)
    qv = estimate_qv(final_data.VCLAY, final_data.PHIT, cec_clay=.1)
    phit_shale = estimate_shale_porosity(final_data.NPHI, final_data.PHIT)
    qvn = estimate_qvn(final_data.VCLAY, final_data.PHIT, phit_shale)
    swt = waxman_smits_saturation(final_data['RT_EDIT'], rw, final_data.PHIE, B=b, Qv=qvn, m=m)

    final_data['SWT'] = swt.clip(0, 1)
    
    # Save result to database
    with db_conn.get_session() as db_session:
        project = Project(db_session, name=project_name)
        project.update_data(final_data)
        project.save()